In [15]:
import random

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE, BatchNorm, GATConv
from torch_geometric.utils import dropout_adj, from_scipy_sparse_matrix
from torch_geometric.data import Data
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import BatchNorm1d

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

import scipy.sparse as sp

import networkx as nx
import matplotlib.pyplot as plt

from tqdm import tqdm

In [17]:
# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

Loaded data with shape: (100, 1700)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 100
Node features shape: (100, 100)


In [156]:
# Functions

# Normalized_laplacian
def normalize_adj(adj):
    """Compute the normalized Laplacian."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])  # Identity matrix of the same size as adj
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)  # Normalized Laplacian
    return laplacian.tocoo()


# ---------------------------
# GAT encoder definition
# ---------------------------
class GATEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3,
                 dropout_rate=0.01, edge_dropout_rate=0.01, heads=3):
        """
        Parameters:
            in_channels (int): Number of features per node.
            hidden_channels (int): Hidden dimension size (per head) used for attention.
            out_channels (int): Final embedding dimension.
            num_layers (int): Total number of graph attention layers.
            dropout_rate (float): Dropout probability applied after each layer.
            edge_dropout_rate (float): Dropout probability applied to the edge_index.
            heads (int): Number of attention heads in each layer.
                        (Default is set so that the first layer returns 2*hidden_channels
                         when concat=True.)
        """
        super(GATEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.edge_dropout_rate = edge_dropout_rate
        self.heads = heads

        self.convs = torch.nn.ModuleList()
        # First layer:
        # Using concat=True returns output of shape (N, hidden_channels * heads).
        self.convs.append(
            GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout_rate, concat=True)
        )

        # Intermediate layers:
        for _ in range(num_layers - 2):
            self.convs.append(
                GATConv(hidden_channels * heads, hidden_channels, heads=heads, dropout=dropout_rate, concat=True)
            )

        # Final layer: set concat=False to reduce dimension to out_channels.
        self.convs.append(
            GATConv(hidden_channels * heads, out_channels, heads=1, dropout=dropout_rate, concat=False)
        )

        # Batch normalization layers for each intermediate GAT layer (except the final one)
        self.bns = torch.nn.ModuleList([
            BatchNorm1d(hidden_channels * heads, momentum=0.1) for _ in range(num_layers - 1)
        ])

        self.dropout = torch.nn.Dropout(p=dropout_rate)
        self.activation_hidden = torch.nn.ReLU  # For hidden layers
        self.activation_output = torch.nn.Sigmoid()  # For final output activation

    def forward(self, x, edge_index):
        # Apply edge dropout
        edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)

        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        x = F.relu(x)

        # Intermediate layers: batch normalization, dropout, and residual connections.
        for i in range(1, self.num_layers - 1):
            x_residual = x  # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i - 1](x)  # Apply batch normalization
            x = F.relu(x)
            x = self.dropout(x)
            # Residual/skip connection
            x = x + x_residual

        # Final layer (no batch norm or dropout)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        # Optionally, you could use a sigmoid activation here with:
         x = self.activation_output(x)
        return x


# ---------------------------
# GAE model decoder and training
# ---------------------------
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=16,
                           epochs=200, dropout_rate=0.5, edge_dropout_rate=0.5, weight_decay=1e-6):
    """
    Apply a Graph Attention-based Autoencoder on an adjacency matrix.
    
    Parameters:
        adj_matrix (np.array or scipy.sparse matrix): The input adjacency matrix.
        node_features (np.array): Node feature matrix.
        embedding_dim (int): Dimension for both hidden and output embedding.
        epochs (int): Number of training epochs.
        dropout_rate (float): Dropout rate for the network layers.
        edge_dropout_rate (float): Dropout rate for the edges.
        weight_decay (float): Weight decay (L2 regularization) for the optimizer.
    
    Returns:
        normalized_adj (np.array): Reconstructed and normalized adjacency matrix.
    """
    # Convert adjacency matrix to sparse format if needed
    if not sp.isspmatrix(adj_matrix):
        adj_sparse = sp.csr_matrix(adj_matrix)
    else:
        adj_sparse = adj_matrix

    # Normalize the adjacency matrix (you may need to implement or import normalize_adj)
    # Here, assume normalize_adj is defined elsewhere:
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    
    # Get edge_index and edge_weight from the normalized sparse matrix
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GATEncoder as the encoder for the GAE model
    encoder = GATEncoder(
        in_channels=x.shape[1],
        hidden_channels=embedding_dim,
        out_channels=embedding_dim,
        dropout_rate=dropout_rate,
        edge_dropout_rate=edge_dropout_rate,
        heads=2  # you can change this parameter as needed
    )
    
    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Set up training
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)

    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # After training, reconstruct the adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
        
    # Normalize the reconstructed adjacency matrix to be between 0 and 1
    adj_min = adj_reconstructed.min()
    adj_max = adj_reconstructed.max()
    normalized_adj = (adj_reconstructed - adj_min) / (adj_max - adj_min)
    
    return normalized_adj


# ---------------------------
# Running GAT-GAE multiple times
# ---------------------------
def run_gae_with_algo_filtering(data, algorithms, thresholds, num_iterations=2, algo_threshold=2):
    """
    Run GAE (with GAT encoder) multiple times per algorithm, apply filtering across iterations per algorithm,
    and return only the filtered binary adjacency matrices per algorithm.
    
    Parameters:
        data (pd.DataFrame): full input matrix (shape: num_genes × num_algorithms × genes)
        algorithms (list): list of algorithm prefixes like "GRN_PCIT"
        thresholds (list): list of float binarization thresholds (e.g. [0.4])
        num_iterations (int): number of repeated GAE runs per algorithm
        algo_threshold (int): number of iterations an edge must appear in to be kept
        
    Returns:
        filtered_results (dict): keys are thresholds and inside each algorithm, the filtered matrix (e.g. 100×100)
    """
    num_genes = data.shape[0]
    filtered_results = {}

    for threshold in thresholds:
        print(f"\n🔹 Running GAE with binarization threshold: {threshold}")
        filtered_results[f'threshold_{threshold}'] = {}

        for alg in algorithms:
            print(f"  🔸 Algorithm: {alg}")
            alg_columns = [col for col in data.columns if col.startswith(alg + '.')]

            if len(alg_columns) != num_genes:
                print(f"    ⚠️ Skipping {alg}: expected {num_genes} columns, got {len(alg_columns)}")
                continue  # Skip malformed input

            alg_matrix = data[alg_columns].values.reshape((num_genes, num_genes))
            matrices = []

            for iteration in range(num_iterations):
                print(f"    ⏳ Iteration {iteration + 1}/{num_iterations}")
                adj_reconstructed = apply_gae_to_adjacency(alg_matrix, node_features)
                adj_binary = (adj_reconstructed > threshold).astype(int)
                matrices.append(adj_binary)

            # Stack and filter: keep edges that appear in at least `algo_threshold` iterations
            matrices = np.stack(matrices, axis=0)  # shape: (num_iterations, num_genes, num_genes)
            filtered = (np.sum(matrices, axis=0) >= algo_threshold).astype(int)
            filtered_results[f'threshold_{threshold}'][alg] = filtered

    return filtered_results



IndentationError: unexpected indent (3301082130.py, line 89)

In [158]:
# Define thresholds and number of iterations
thresholds = [0]  # Different thresholds to run
num_iterations = 10  # Number of runs for each threshold
algo_threshold=4 # how many algorithm 

# Run the GAE process and store results for each threshold and algorithm
filtered_results = run_gae_with_algo_filtering(
    data=data,
    algorithms=algorithms,
    thresholds=thresholds,
    num_iterations=num_iterations,
    algo_threshold=algo_threshold
)




🔹 Running GAE with binarization threshold: 0
  🔸 Algorithm: GRN_ARACNE
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10
  🔸 Algorithm: GRN_BC3NET
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10
  🔸 Algorithm: GRN_C3NET
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


... (27 outputs left out)

/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10


/var/folders/fc/x_bs_jtd3x19_tzx4mjmg3xc0000gn/T/ipykernel_76441/361845540.py:166: RuntimeWarning: invalid value encountered in divide
  normalized_adj = (adj_reconstructed - adj_min) / (adj_max - adj_min)
/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


  🔸 Algorithm: GRN_TIGRESS
    ⏳ Iteration 1/10
    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10


In [159]:
# Applying NMF to 17 algorithm
def fuse_with_joint_nmf_from_filtered(filtered_results, threshold, rank=20, num_genes=100):
    """
    Fuses multiple filtered binary adjacency matrices using joint NMF.
    Each matrix must be (num_genes x num_genes), one per algorithm.
    """
    all_flattened = []

    for alg, matrix in filtered_results[f'threshold_{threshold}'].items():
        assert matrix.shape == (num_genes, num_genes), f"{alg} shape is invalid: {matrix.shape}"
        flattened = matrix.flatten()
        all_flattened.append(flattened)

    # all_flattened = 17 x 10000 → each row is one algorithm's flattened adjacency matrix
    nmf_input = np.array(all_flattened)  # shape: (17, 10000)
    print(f"Matrix shape (before transpose): {nmf_input.shape}")

    V = nmf_input.T  # (10000, 17)
    print(f"NMF input shape: {V.shape}")

    nmf_model = NMF(n_components=rank, init='random', random_state=42, max_iter=1000)
    W = nmf_model.fit_transform(V)
    H = nmf_model.components_

    # Reconstruct the fused graph
    V_reconstructed = np.dot(W, H).mean(axis=1)  # shape: (10000,)

    # RESHAPE using known num_genes = 100, not inferred!
    fused_adj = V_reconstructed.reshape((num_genes, num_genes))

    # Normalize to [0, 1]
    fused_adj = (fused_adj - fused_adj.min()) / (fused_adj.max() - fused_adj.min())

    return fused_adj



In [160]:
# Running the NMF fusion
fused_adj_nmf = fuse_with_joint_nmf_from_filtered(filtered_results, threshold=0, rank=20)


Matrix shape (before transpose): (17, 10000)
NMF input shape: (10000, 17)


In [161]:
# Function to extract nodes and edges from the binary matrix
def count_nodes_edges(final_matrix):
    # Count edges in the upper triangle (to avoid double counting for undirected graphs)
    num_edges = np.sum(np.triu(final_matrix, k=1))
    
    # Count nodes (genes with at least one edge, i.e., rows/columns with non-zero values)
    connected_nodes = np.sum(np.any(final_matrix, axis=1))  # Count rows with non-zero values
    num_nodes = connected_nodes
    
    return num_nodes, num_edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


In [162]:
threshold = 0.4
fused_binary = (fused_adj_nmf >= threshold).astype(int)

In [163]:
# Count nodes and edges
num_nodes, num_edges = count_nodes_edges(fused_binary)
print(f"Nodes: {num_nodes}, Edges: {num_edges}")

# Get the edge list
nmf_edge_list = extract_gene_interactions(fused_binary, gene_names)
print(f"Sample edges: {nmf_edge_list[:5]}")

Nodes: 99, Edges: 805
Sample edges: ['G1-G18', 'G1-G22', 'G1-G23', 'G1-G24', 'G1-G25']


In [164]:
# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")


Ground truth loaded with shape: (100, 100)
Number of known edges: 175


In [165]:
def convert_str_edges_to_tuples(edge_list):
    edge_tuples = set()
    for edge in edge_list:
        a, b = edge.split('-')
        # Add both (a, b) and (b, a) to ensure directionality doesn't affect matching
        edge_tuples.add((a, b))
        edge_tuples.add((b, a))
    return edge_tuples

predicted_edge_set = convert_str_edges_to_tuples(nmf_edge_list)

def symmetrize_edges(edge_set):
    symmetric_set = set()
    for a, b in edge_set:
        symmetric_set.add((a, b))
        symmetric_set.add((b, a))
    return symmetric_set

known_edges_symmetric = symmetrize_edges(known_edges_set)



In [166]:
def convert_str_edges_to_tuples(edge_list):
    edge_tuples = set()
    for edge in edge_list:
        a, b = edge.split('-')
        # Add both (a, b) and (b, a) to ensure directionality doesn't affect matching
        edge_tuples.add((a, b))
        edge_tuples.add((b, a))
    return edge_tuples

predicted_edge_set = convert_str_edges_to_tuples(nmf_edge_list)

def symmetrize_edges(edge_set):
    symmetric_set = set()
    for a, b in edge_set:
        symmetric_set.add((a, b))
        symmetric_set.add((b, a))
    return symmetric_set

known_edges_symmetric = symmetrize_edges(known_edges_set)



In [167]:
# True Positives (correct predictions)
true_positives = predicted_edge_set.intersection(known_edges_symmetric)

# False Positives (predicted but not known)
false_positives = predicted_edge_set - known_edges_symmetric

# False Negatives (missed known edges)
false_negatives = known_edges_symmetric - predicted_edge_set

# Precision, Recall, F1
tp = len(true_positives) // 2  # divide by 2 because of symmetric edges
fp = len(false_positives) // 2
fn = len(false_negatives) // 2

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")


TP: 72, FP: 733, FN: 96
Precision: 0.0894, Recall: 0.4286, F1 Score: 0.1480


In [168]:
#### AUPR and AUROC

from sklearn.metrics import roc_auc_score, average_precision_score

def compute_auroc_aupr(predicted_adj, ground_truth_df, gene_names):
    """
    Computes AUROC and AUPR given predicted adjacency (scores) and ground truth (binary).
    Assumes both are 100x100 matrices with matching gene order.
    """

    # Convert ground truth DataFrame to a numpy binary matrix
    gt_matrix = ground_truth_df.loc[gene_names, gene_names].values.astype(int)

    # Flatten only upper triangle (excluding diagonal)
    triu_indices = np.triu_indices_from(gt_matrix, k=1)
    gt_flat = gt_matrix[triu_indices]
    pred_flat = predicted_adj[triu_indices]

    # AUROC and AUPR
    auroc = roc_auc_score(gt_flat, pred_flat)
    aupr = average_precision_score(gt_flat, pred_flat)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPR : {aupr:.4f}")
    
    return auroc, aupr


In [169]:
# Call the scorer
compute_auroc_aupr(fused_adj_nmf, ground_truth, gene_names)

AUROC: 0.7664
AUPR : 0.0790


(0.766395880308208, 0.0789949472343828)

In [114]:
# Call the scorer
compute_auroc_aupr(fused_adj_nmf, ground_truth, gene_names)

AUROC: 0.8099
AUPR : 0.0795


(0.8098566385250889, 0.07951982428447467)